# 01 — Data Diagnosis
**Project:** excel-data-cleaner  
**Purpose:** Understand data quality issues before writing any cleaning rule.  
**Input:** `data/raw/dirty_sales_data.xlsx`

> This notebook answers four questions:
> 1. Structure — what columns exist and how are they named?
> 2. Quality — how many nulls, empties, duplicates?
> 3. Consistency — are values clean in categorical/date/numeric columns?
> 4. Business rules — what records are clearly invalid?

---

# 01 — Diagnóstico de Datos
**Proyecto:** excel-data-cleaner  
**Objetivo:** Comprender los problemas de calidad de los datos antes de definir cualquier regla de limpieza.  
**Entrada:** `data/raw/dirty_sales_data.xlsx`

> Este notebook responde a cuatro preguntas:
> 1. Estructura — ¿qué columnas existen y cómo están nombradas?
> 2. Calidad — ¿cuántos valores nulos, vacíos o duplicados hay?
> 3. Consistencia — ¿los valores son coherentes en columnas categóricas, de fecha y numéricas?
> 4. Reglas de negocio — ¿qué registros son claramente inválidos?

## 0 — Setup

Import required libraries and define the path to the raw dataset.

---

## 0 — Configuración

Importar las librerías necesarias y definir la ruta al dataset original.

In [1]:
import pandas as pd
import sys
from pathlib import Path
from datetime import datetime

print("Libraries loaded successfully")
print(f"Validation timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ___ Excel file loading / Lectura del archivo Excel _________________________________

# Funciona tanto si ejecutas desde /notebooks como desde la raíz del proyecto

ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INPUT_FILE = ROOT / "data" / "raw" / "dirty_sales_data.xlsx"

df = pd.read_excel(INPUT_FILE, dtype=str)  # Read all columns as string to avoid early type coercion
print(f"File loaded: {INPUT_FILE.name}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(5)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

Libraries loaded successfully
Validation timestamp: 2026-03-21 11:44:21

File loaded: dirty_sales_data.xlsx
Shape: 320 rows × 14 columns


## 1 — Structure

> What columns are present in the dataset?  
> How are they named?  
> Are there hidden issues in the column headers?

---

## 1 — Estructura

> ¿Qué columnas hay en el dataset?  
> ¿Cómo están nombradas?  
> ¿Existen problemas ocultos en los encabezados?

In [2]:

print("\n___ Columns overview: null counts and data types ___\n")

print(f"Columns: {list(df.columns)}\n")
print(f"Shape: {df.shape}\n")

df.info()

print("\nMissing values per column:\n")
print(df.isna().sum())





___ Columns overview: null counts and data types ___

Columns: [' ID Venta ', 'cliente', 'Cliente', 'fecha venta', 'Fecha_Venta', 'producto ', 'tipo_madera', 'certificacion', 'cantidad_m3 ', 'precio_m3', 'importe ', 'estado', 'comercial', 'pais']

Shape: (320, 14)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0    ID Venta      320 non-null    object
 1   cliente        254 non-null    object
 2   Cliente        105 non-null    object
 3   fecha venta    320 non-null    object
 4   Fecha_Venta    165 non-null    object
 5   producto       320 non-null    object
 6   tipo_madera    320 non-null    object
 7   certificacion  230 non-null    object
 8   cantidad_m3    249 non-null    object
 9   precio_m3      320 non-null    object
 10  importe        184 non-null    object
 11  estado         320 non-null    object
 12  comercial      320 no

In [19]:
print("\n___ Column header issues ___\n")

for i, col in enumerate(df.columns):
    has_leading_space  = col != col.lstrip()
    has_trailing_space = col != col.rstrip()
    has_inner_space    = " " in col.strip()
    
    issues = []
    
    if has_leading_space:
        issues.append("leading space")
        
    if has_trailing_space:
        issues.append("trailing space")
        
    if has_inner_space:
        issues.append("inner space (may affect naming consistency)")
        
    issues_str = f"  ⚠ {', '.join(issues)}" if issues else ""
    
    print(f"[{i}] '{col}'{issues_str}")


___ Column header issues ___

[0] ' ID Venta '  ⚠ leading space, trailing space, inner space (may affect naming consistency)
[1] 'cliente'
[2] 'Cliente'
[3] 'fecha venta'  ⚠ inner space (may affect naming consistency)
[4] 'Fecha_Venta'
[5] 'producto '  ⚠ trailing space
[6] 'tipo_madera'
[7] 'certificacion'
[8] 'cantidad_m3 '  ⚠ trailing space
[9] 'precio_m3'
[10] 'importe '  ⚠ trailing space
[11] 'estado'
[12] 'comercial'
[13] 'pais'


In [4]:
print("\n___ Duplicate columns after normalization ___\n")

normalized_headers = [col.strip().lower().replace(" ", "_") for col in df.columns]

seen = {}
duplicates = []

for original_col, norm_col in zip(df.columns, normalized_headers):
    if norm_col in seen:
        print(f"'{original_col}' normalized as '{norm_col}' (same as '{seen[norm_col]}') → ⚠ DUPLICATE")
        duplicates.append((original_col, seen[norm_col], norm_col))
    else:
        seen[norm_col] = original_col
        print(f"'{original_col}' → '{norm_col}' → OK")
# Note: we are not modifying df yet, only inspecting structure


___ Duplicate columns after normalization ___

' ID Venta ' → 'id_venta' → OK
'cliente' → 'cliente' → OK
'Cliente' normalized as 'cliente' (same as 'cliente') → ⚠ DUPLICATE
'fecha venta' → 'fecha_venta' → OK
'Fecha_Venta' normalized as 'fecha_venta' (same as 'fecha venta') → ⚠ DUPLICATE
'producto ' → 'producto' → OK
'tipo_madera' → 'tipo_madera' → OK
'certificacion' → 'certificacion' → OK
'cantidad_m3 ' → 'cantidad_m3' → OK
'precio_m3' → 'precio_m3' → OK
'importe ' → 'importe' → OK
'estado' → 'estado' → OK
'comercial' → 'comercial' → OK
'pais' → 'pais' → OK


## 2 — Data Quality

> Are there missing values, empty rows or duplicated records?  
> Are data types consistent with the expected structure for analysis?

---

## 2 — Calidad de datos

> ¿Existen valores nulos, filas vacías o registros duplicados?  
> ¿Son los tipos de datos coherentes con lo esperado para el análisis?

In [5]:
print("\n___ Missing values analysis ___\n")

def analyze_missing_values(df):
    
    results = []
    total = len(df)
    
    for column in df.columns:
        non_null_count = df[column].notna().sum()
        null_count = total - non_null_count
        null_percentage = (null_count / total) * 100
        
        results.append({
            'column': column,
            'non_null_count': non_null_count,
            'null_count': null_count,
            'null_percentage': round(null_percentage, 2)
        })
    
    result_df = pd.DataFrame(results)
    return result_df
    
# Execute analysis
df_missing = analyze_missing_values(df)

print(df_missing.to_string(index=False))


___ Missing values analysis ___

       column  non_null_count  null_count  null_percentage
    ID Venta              320           0             0.00
      cliente             254          66            20.62
      Cliente             105         215            67.19
  fecha venta             320           0             0.00
  Fecha_Venta             165         155            48.44
    producto              320           0             0.00
  tipo_madera             320           0             0.00
certificacion             230          90            28.12
 cantidad_m3              249          71            22.19
    precio_m3             320           0             0.00
     importe              184         136            42.50
       estado             320           0             0.00
    comercial             320           0             0.00
         pais             320           0             0.00


In [6]:
print("\n___ Empty rows analysis ___\n")

empty_rows = df.isna().all(axis=1).sum()
print(f"Empty rows: {empty_rows}")


___ Empty rows analysis ___

Empty rows: 0


In [7]:
total_rows = len(df)
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows} ({(duplicate_rows/total_rows)*100:.2f}%)")

Duplicate rows: 20 (6.25%)


In [8]:
# Apparent data types (everything is str because dtype=str)
# We infer what EACH column SHOULD be

print("\n___ Apparent data types (based on values) ___\n")

for col in df.columns:
    examples = df[col].dropna().head(5).tolist()
    print(f"  {col!r:25s}  sample of 5 elements: {examples}")


___ Apparent data types (based on values) ___

  ' ID Venta '               sample of 5 elements: ['1000', '1001', '1002', '1003', '1004']
  'cliente'                  sample of 5 elements: ['Obras y Casas', 'Construcciones Norte', 'Muebles Pérez Polo', 'Promociones Sur', 'Muebles Pérez Polo']
  'Cliente'                  sample of 5 elements: ['Obras y Casas', 'Construcciones Norte', 'Promociones Sur', 'Construcciones Norte', 'Muebles Pérez Polo']
  'fecha venta'              sample of 5 elements: ['01/02/2024', '2024/02/05', '2024/02/05', '01/02/2024', '2024/02/05']
  'Fecha_Venta'              sample of 5 elements: ['2024-02-06', '2024-02-06', '2024-02-06', '2024-02-06', '2024-02-06']
  'producto '                sample of 5 elements: ['Viga', 'CLT', 'Viga laminada', 'CLT', 'CLT']
  'tipo_madera'              sample of 5 elements: ['Pino', 'Abeto', 'Abeto', 'abeto', 'PINO']
  'certificacion'            sample of 5 elements: ['PEFC', 'CE', 'CE', 'CE', 'CE']
  'cantidad_m3 '         

## 3 — Consistency

> Unique values in categorical columns, date format issues, numeric issues.

---

## 3 — Consistencia

> Valores únicos en columnas categóricas, problemas de formato de fecha, problemas numéricos.

In [9]:
print("___ Unique values in key columns ___\n")

print(f"Number of duplicated sales IDs: {df[' ID Venta '].duplicated().sum()}\n")

key_columns = ["cliente", "Cliente", "fecha venta", "Fecha_Venta", "importe ", "cantidad_m3 "]

for col in key_columns:
    print(f"In key column {col}: {list(df[col].unique())}")
    


___ Unique values in key columns ___

Number of duplicated sales IDs: 20

In key column cliente: ['Obras y Casas', 'Construcciones Norte', nan, 'Muebles Pérez Polo', 'Promociones Sur', 'Reformas López']
In key column Cliente: [nan, 'Obras y Casas', 'Construcciones Norte', 'Promociones Sur', 'Muebles Pérez Polo', 'Reformas López']
In key column fecha venta: ['01/02/2024', '2024/02/05', '2024-02-03', '03-02-2024']
In key column Fecha_Venta: [nan, '2024-02-06']
In key column importe : ['675', nan, '1125', '480', '900', '640', '800']
In key column cantidad_m3 : ['1.5', nan, '2.5', '2', '0']


In [10]:
print("___ Inconsistent categories (unique values in categorical columns) ___\n")

categorical_columns = ["cliente", "Cliente", "producto ", "tipo_madera", "certificacion", "estado", "comercial", "pais"]

for c in categorical_columns:
    unique_vals = sorted(df[c].dropna().unique())
    print(f"--- {c} ({len(unique_vals)}) unique values ---")

    for v in unique_vals:
        count = (df[c] == v).sum()
        print(f"{count:3d} X '{v}'")
    print()

___ Inconsistent categories (unique values in categorical columns) ___

--- cliente (5) unique values ---
 57 X 'Construcciones Norte'
 49 X 'Muebles Pérez Polo'
 44 X 'Obras y Casas'
 47 X 'Promociones Sur'
 57 X 'Reformas López'

--- Cliente (5) unique values ---
 25 X 'Construcciones Norte'
 20 X 'Muebles Pérez Polo'
 25 X 'Obras y Casas'
 18 X 'Promociones Sur'
 17 X 'Reformas López'

--- producto  (3) unique values ---
131 X 'CLT'
105 X 'Viga'
 84 X 'Viga laminada'

--- tipo_madera (4) unique values ---
 86 X 'Abeto'
 89 X 'PINO'
 65 X 'Pino'
 80 X 'abeto'

--- certificacion (3) unique values ---
 90 X 'CE'
 88 X 'FSC'
 52 X 'PEFC'

--- estado (5) unique values ---
 61 X 'CERRADO'
 65 X 'Cerrada'
 73 X 'cerrado'
 61 X 'ok'
 60 X 'pendiente'

--- comercial (2) unique values ---
150 X 'Ana'
170 X 'Juan'

--- pais (3) unique values ---
 93 X 'ES'
 98 X 'España'
129 X 'Madrid'



In [11]:
print("\n___ Date formats ___\n")

date_columns = ["fecha venta", "Fecha_Venta"]
df_sales_date = df[["fecha venta", "Fecha_Venta"]]

for c in df_sales_date:    
    unique_vals = sorted(df[c].dropna().unique())
    print(f"--- {c} ({len(unique_vals)}) unique values ---")

    for v in unique_vals:
        count = (df[c] == v).sum()
        print(f"{count:3d} X '{v}'")
    print()



___ Date formats ___

--- fecha venta (4) unique values ---
 94 X '01/02/2024'
 57 X '03-02-2024'
 76 X '2024-02-03'
 93 X '2024/02/05'

--- Fecha_Venta (1) unique values ---
165 X '2024-02-06'



In [12]:
print("\n___ Numeric columns format ___\n")

print('''The Excel file was loaded as string to avoid Pandas inferring data types and altering the general analysis.
However, this limits the exploration of numeric columns in this notebook.
Data types will be validated in validation.py''')


print("\n___ Formato columnas numéricas ___\n")

print('''El Excel fue cargado como string para evitar que Pandas infiera tipos y el análisis general quede alterado.
Pero limita la exploración de las columnas numéricas en este notebook.
Validar tipos en validation.py''')


___ Numeric columns format ___

The Excel file was loaded as string to avoid Pandas inferring data types and altering the general analysis.
However, this limits the exploration of numeric columns in this notebook.
Data types will be validated in validation.py

___ Formato columnas numéricas ___

El Excel fue cargado como string para evitar que Pandas infiera tipos y el análisis general quede alterado.
Pero limita la exploración de las columnas numéricas en este notebook.
Validar tipos en validation.py


## 4 — Business Rules

> Which records should clearly not exist?  
> Which columns are mandatory?  
> Which values are invalid?

---

## 4 — Reglas de negocio

> ¿Qué registros claramente no deberían existir?  
> ¿Qué columnas son obligatorias?  
> ¿Qué valores son inválidos?

In [13]:
print("\n___ Which records should not exist?: check how many rows have null values in key columns ___ \n")

key_columns = [" ID Venta ", "cliente", "Cliente", "fecha venta", "Fecha_Venta", "importe ", "cantidad_m3 "]

for col in key_columns:
    if col in df.columns:
        nulls = df[col].isna().sum()
        status = f"⚠ {nulls} nulls" if nulls else "✓ complete"
        print(f"{col:15s}: {status}")
        #print(f"{col:20s} {status}")


___ Which records should not exist?: check how many rows have null values in key columns ___ 

 ID Venta      : ✓ complete
cliente        : ⚠ 66 nulls
Cliente        : ⚠ 215 nulls
fecha venta    : ✓ complete
Fecha_Venta    : ⚠ 155 nulls
importe        : ⚠ 136 nulls
cantidad_m3    : ⚠ 71 nulls


In [14]:
print("\n ___ Duplicate sales IDs ___\n")

duplicate_ids = df[df.duplicated(' ID Venta ', keep='first')][' ID Venta '].dropna()

print(f"Total duplicates: {len(duplicate_ids)}\n")

for v in duplicate_ids.unique():
    count = (df[' ID Venta '] == v).sum()
    print(f"{count:3d} X '{v}'")



 ___ Duplicate sales IDs ___

Total duplicates: 20

  2 X '1203'
  2 X '1266'
  2 X '1152'
  2 X '1009'
  2 X '1233'
  2 X '1226'
  2 X '1196'
  2 X '1109'
  2 X '1005'
  2 X '1175'
  2 X '1237'
  2 X '1057'
  2 X '1218'
  2 X '1045'
  2 X '1182'
  2 X '1221'
  2 X '1289'
  2 X '1211'
  2 X '1148'
  2 X '1165'


In [15]:
print("\n ___ Invalid values ___\n")

df = pd.read_excel('../data/raw/dirty_sales_data.xlsx', dtype=str)

df_norm = df.copy()
df_norm["cliente"] = df["cliente"].combine_first(df["Cliente"])
df_norm["fecha venta"] = df["fecha venta"].combine_first(df["Fecha_Venta"])
df_norm = df_norm.drop(columns=["Cliente", "Fecha_Venta"])
df_norm.columns = [c.strip().lower().replace(" ","_") for c in df_norm.columns]

numeric_importe = pd.to_numeric(df_norm["importe"], errors="coerce") if "importe" in df_norm.columns else None
parsed_fecha    = pd.to_datetime(df_norm["fecha_venta"], errors="coerce") if "fecha_venta" in df_norm.columns else None

VALID_ESTADOS = {"CERRADO", "Cerrada", "ok", "pendiente"}
invalid_estado = ~df_norm["estado"].str.lower().isin(VALID_ESTADOS) if "estado" in df_norm.columns else pd.Series(False)

rules = {
    "id_venta is null"           : df_norm["id_venta"].isna() if "id_venta" in df_norm.columns else 0,
    "fecha_venta is null"        : parsed_fecha.isna() if parsed_fecha is not None else 0,
    "importe is null or <= 0"    : (numeric_importe.isna() | (numeric_importe <= 0)) if numeric_importe is not None else 0,
    "estado not in valid set"    : invalid_estado,
}

for rule, mask in rules.items():
    count = mask.sum() if hasattr(mask, "sum") else mask
    flag  = "⚠" if count > 0 else "✓"
    print(f"  {flag}  {rule:35s} {count} rows")



 ___ Invalid values ___

  ✓  id_venta is null                    0 rows
  ⚠  fecha_venta is null                 226 rows
  ⚠  importe is null or <= 0             136 rows
  ⚠  estado not in valid set             199 rows


⚠ Note: fecha_venta shows 226 nulls because pd.to_datetime() does not recognize the 4 mixed date formats.  
The actual null values are 0 — confirmed in the previous missing values analysis block.  
Multi-format date conversion will be handled in cleaning.py.


⚠ Nota: fecha_venta muestra 226 nulls porque pd.to_datetime() 
no reconoce los 4 formatos mezclados.  
Los nulls reales son 0 — 
confirmado en el bloque de análisis de nulos anterior.  La conversión multi-formato se aplica en cleaning.py.

## 5 — Diagnosis Summary

> Quick reference before defining data cleaning rules.

---

## 5 — Resumen del diagnóstico

> Referencia rápida antes de escribir reglas de limpieza.

In [16]:
print("=" * 52)
print("  DATA DIAGNOSIS SUMMARY")
print("=" * 52)
print(f"  Total rows            : {len(df)}")
print(f"  Columns               : {len(df.columns)}")
print()
print("  Issues detected:")
print(f"    Fully empty rows    : {df.isna().all(axis=1).sum()}")
print(f"    Exact duplicates    : {df.duplicated().sum()}")
print(f"    Duplicate id_venta  : {df.duplicated(subset=[' ID Venta '], keep=False).sum() // 2 if ' ID Venta ' in df.columns else 'N/A'}")
print("  (* cliente/Cliente y fecha venta/Fecha_Venta son columnas duplicadas)")
print()
print("  Columns with nulls:")
for col in df.columns:
    n = df[col].isna().sum()
    if n > 0:
        print(f"    {col:20s} {n} nulls  ({n/len(df)*100:.0f}%)")
print()
print("  Cleaning rules to apply:")
print("    ✓ Normalize column names")
print("    ✓ Drop fully empty rows")
print("    ✓ Remove duplicates by id_venta")
print("    ✓ Standardize: cliente, comercial, pais, tipo_madera (Title Case)")
print("    ✓ Map estado → canonical values (Cerrado, Pendiente, Cancelado)")
print("    ✓ Parse fecha_venta to datetime")
print("    ✓ Parse importe to numeric")
print("    ✓ Filter: null id, null date, importe <= 0")
print("=" * 52)

  DATA DIAGNOSIS SUMMARY
  Total rows            : 320
  Columns               : 14

  Issues detected:
    Fully empty rows    : 0
    Exact duplicates    : 20
    Duplicate id_venta  : 20
  (* cliente/Cliente y fecha venta/Fecha_Venta son columnas duplicadas)

  Columns with nulls:
    cliente              66 nulls  (21%)
    Cliente              215 nulls  (67%)
    Fecha_Venta          155 nulls  (48%)
    certificacion        90 nulls  (28%)
    cantidad_m3          71 nulls  (22%)
    importe              136 nulls  (42%)

  Cleaning rules to apply:
    ✓ Normalize column names
    ✓ Drop fully empty rows
    ✓ Remove duplicates by id_venta
    ✓ Standardize: cliente, comercial, pais, tipo_madera (Title Case)
    ✓ Map estado → canonical values (Cerrado, Pendiente, Cancelado)
    ✓ Parse fecha_venta to datetime
    ✓ Parse importe to numeric
    ✓ Filter: null id, null date, importe <= 0


In [17]:
print("=" * 52)
print("  DATA DIAGNOSIS SUMMARY")
print("  RESUMEN DEL DIAGNÓSTICO")
print("=" * 52)

print(f"  Total rows / Total filas : {len(df)}")
print(f"  Columns / Columnas       : {len(df.columns)}")
print()

print("  Issues detected / Problemas detectados:")
print(f"    Fully empty rows / Filas vacías        : {df.isna().all(axis=1).sum()}")
print(f"    Exact duplicates / Duplicados exactos  : {df.duplicated().sum()}")
print(f"    Duplicate id_venta / ID duplicados     : {df.duplicated(subset=[' ID Venta '], keep=False).sum() // 2 if ' ID Venta ' in df.columns else 'N/A'}")
print("  (* cliente/Cliente and fecha venta/Fecha_Venta are duplicated columns)")
print("  (* cliente/Cliente y fecha venta/Fecha_Venta son columnas duplicadas)")
print()

print("  Columns with nulls / Columnas con nulos:")
for col in df.columns:
    n = df[col].isna().sum()
    if n > 0:
        print(f"    {col:20s} {n} nulls / nulos  ({n/len(df)*100:.0f}%)")
print()

print("  Cleaning rules to apply / Reglas de limpieza:")
print("    ✓ Normalize column names / Normalizar nombres de columnas")
print("    ✓ Drop fully empty rows / Eliminar filas completamente vacías")
print("    ✓ Remove duplicates by id_venta / Eliminar duplicados por id_venta")
print("    ✓ Standardize: cliente, comercial, pais, tipo_madera (Title Case)")
print("      / Estandarizar: cliente, comercial, pais, tipo_madera (Title Case)")
print("    ✓ Map estado → canonical values (Cerrado, Pendiente, Cancelado)")
print("      / Mapear estado → valores canónicos (Cerrado, Pendiente, Cancelado)")
print("    ✓ Parse fecha_venta to datetime / Convertir fecha_venta a datetime")
print("    ✓ Parse importe to numeric / Convertir importe a numérico")
print("    ✓ Filter: null id, null date, importe <= 0")
print("      / Filtrar: id nulo, fecha nula, importe <= 0")

print("=" * 52)

  DATA DIAGNOSIS SUMMARY
  RESUMEN DEL DIAGNÓSTICO
  Total rows / Total filas : 320
  Columns / Columnas       : 14

  Issues detected / Problemas detectados:
    Fully empty rows / Filas vacías        : 0
    Exact duplicates / Duplicados exactos  : 20
    Duplicate id_venta / ID duplicados     : 20
  (* cliente/Cliente and fecha venta/Fecha_Venta are duplicated columns)
  (* cliente/Cliente y fecha venta/Fecha_Venta son columnas duplicadas)

  Columns with nulls / Columnas con nulos:
    cliente              66 nulls / nulos  (21%)
    Cliente              215 nulls / nulos  (67%)
    Fecha_Venta          155 nulls / nulos  (48%)
    certificacion        90 nulls / nulos  (28%)
    cantidad_m3          71 nulls / nulos  (22%)
    importe              136 nulls / nulos  (42%)

  Cleaning rules to apply / Reglas de limpieza:
    ✓ Normalize column names / Normalizar nombres de columnas
    ✓ Drop fully empty rows / Eliminar filas completamente vacías
    ✓ Remove duplicates by id_venta